# `TimeSeries` Class — Showcase

This notebook demonstrates the `TimeSeries` class from `timedatamodel.timeseries_arrow`.  
It is a PyArrow-backed container that stores time series data together with rich metadata.

**Contents**
1. Imports & setup
2. Helper types — `GeoLocation`, `DataType`, `TimeSeriesType`, `DataShape`
3. Shape 1 — `SIMPLE`: one value per timestamp
4. Shape 2 — `VERSIONED`: forecast revisions (one row per knowledge_time × valid_time)
5. Shape 3 — `AUDIT`: full bi-temporal log (knowledge_time × change_time × valid_time)
6. Metadata & `metadata_dict()`
7. `from_arrow()` — zero-copy path from PyArrow
8. `to_pandas()` round-trip
9. Timezone handling
10. Performance — ingesting a large DataFrame

## 1. Imports & Setup

In [1]:
import sys
import warnings
import time
import numpy as np
import pandas as pd
import pyarrow as pa

# Make sure the package is importable from the repo root
sys.path.insert(0, "..")

from timedb import TimeSeries, DataShape, DataType, TimeSeriesType, GeoLocation

print("PyArrow version :", pa.__version__)
print("Pandas version  :", pd.__version__)

PyArrow version : 23.0.1
Pandas version  : 3.0.1


## 2. Helper Types

Four small types carry all the semantic context for a series.

| Type | Purpose |
|---|---|
| `GeoLocation` | Latitude + longitude of the asset |
| `DataType` | Nature of the observation: `OBSERVATION`, `FORECAST`, `ESTIMATION`, … |
| `TimeSeriesType` | Storage model: `FLAT` (immutable facts) or `OVERLAPPING` (versioned) |
| `DataShape` | Which temporal columns are present: `SIMPLE`, `VERSIONED`, `AUDIT` |

In [2]:
# Geographic location of the wind farm
turbine_location = GeoLocation(latitude=59.33, longitude=18.07)
print(turbine_location)

# DataType values
print([dt.value for dt in DataType])

# TimeSeriesType values
print([tst.value for tst in TimeSeriesType])

# DataShape values (inferred automatically — you never set this directly)
print([ds.value for ds in DataShape])

GeoLocation(latitude=59.33, longitude=18.07)
['ACTUAL', 'OBSERVATION', 'DERIVED', 'CALCULATED', 'ESTIMATION', 'FORECAST', 'PREDICTION', 'SCENARIO', 'SIMULATION', 'RECONSTRUCTION', 'REFERENCE', 'BASELINE', 'BENCHMARK', 'IDEAL']
['FLAT', 'OVERLAPPING']
['SIMPLE', 'VERSIONED', 'AUDIT', 'CORRECTED']


## 3. Shape 1 — `SIMPLE`

A `SIMPLE` series holds one value per timestamp.  
This is the most common shape — it covers:
- **flat** (immutable) series such as meter readings, actuals
- the **latest-value projection** of an overlapping (forecast) series

Required columns: `valid_time`, `value`  
Optional column: `valid_time_end` (for interval-valued data)

`DataShape` is **inferred automatically** from whichever columns are present — you never set it manually.

In [3]:
rng = np.random.default_rng(42)

# ── 3a. Point-in-time values ─────────────────────────────────────────────────
df_actuals = pd.DataFrame({
    "valid_time": pd.date_range("2024-01-01", periods=24, freq="1h", tz="UTC"),
    "value": rng.uniform(50, 120, 24),   # MW
})

ts_actuals = TimeSeries.from_pandas(
    df_actuals,
    name="wind_power",
    unit="MW",
    frequency="1h",
    data_type=DataType.OBSERVATION,
    timeseries_type=TimeSeriesType.FLAT,
    location=GeoLocation(latitude=59.33, longitude=18.07),
    labels={"site": "Gotland", "asset_type": "wind"},
    timezone="Europe/Stockholm",
)

print(ts_actuals)
print(f"  shape     : {ts_actuals.shape}")
print(f"  num_rows  : {ts_actuals.num_rows}")
print(f"  columns   : {ts_actuals.columns}")
print(f"  Arrow schema:")
print(ts_actuals.table.schema)

TimeSeries
┌───────────────────────────────────────────────────────────────┐
│  Name:             wind_power                                 │
│  Shape:            SIMPLE                                     │
│  Rows:             24                                         │
│  Frequency:        1h                                         │
│  Timezone:         Europe/Stockholm                           │
│  Unit:             MW                                         │
│  Data type:        OBSERVATION                                │
│  Location:         59.33°N, 18.07°E                           │
│  Labels:           {'site': 'Gotland', 'asset_type': 'wind'}  │
├───────────────────────────────────────────────────────────────┤
│                    wind_power                                 │
│  2024-01-01 00:00     104.177                                 │
│  2024-01-01 01:00     80.7215                                 │
│  2024-01-01 02:00     110.102                                 │

In [4]:
# ── 3b. from_pandas also accepts valid_time as the DataFrame index ───────────
df_indexed = df_actuals.set_index("valid_time")
ts_from_index = TimeSeries.from_pandas(df_indexed, name="wind_power_from_index", unit="MW")
print(ts_from_index)

# ── 3c. Interval-valued data (valid_time + valid_time_end) ───────────────────
df_energy = pd.DataFrame({
    "valid_time":     pd.date_range("2024-01-01", periods=24, freq="1h", tz="UTC"),
    "valid_time_end": pd.date_range("2024-01-01 01:00", periods=24, freq="1h", tz="UTC"),
    "value": rng.uniform(50, 120, 24),  # MWh
})

ts_energy = TimeSeries.from_pandas(
    df_energy,
    name="wind_energy",
    unit="MWh",
    frequency="1h",
    data_type=DataType.OBSERVATION,
    timeseries_type=TimeSeriesType.FLAT,
)
print(ts_energy)
print(f"  columns : {ts_energy.columns}")

TimeSeries
┌───────────────────────────────────────────┐
│  Name:             wind_power_from_index  │
│  Shape:            SIMPLE                 │
│  Rows:             24                     │
│  Timezone:         UTC                    │
│  Unit:             MW                     │
├───────────────────────────────────────────┤
│                    wind_power_from_index  │
│  2024-01-01 00:00                104.177  │
│  2024-01-01 01:00                80.7215  │
│  2024-01-01 02:00                110.102  │
│  ...                                 ...  │
│  2024-01-01 21:00                74.8168  │
│  2024-01-01 22:00                117.949  │
│  2024-01-01 23:00                112.518  │
└───────────────────────────────────────────┘
TimeSeries
┌─────────────────────────────────┐
│  Name:             wind_energy  │
│  Shape:            SIMPLE       │
│  Rows:             24           │
│  Frequency:        1h           │
│  Timezone:         UTC          │
│  Unit:             MWh  

## 4. Shape 2 — `VERSIONED`

A `VERSIONED` series holds **one row per `(knowledge_time, valid_time)` pair**.  
This is the natural shape for forecast revisions — you can see how a forecast for a given hour changed across successive model runs.

Required columns: `knowledge_time`, `valid_time`, `value`

In [5]:
# Three forecast runs, each covering the same 24-hour valid window
forecast_runs = pd.to_datetime(["2024-01-01 00:00", "2024-01-01 06:00", "2024-01-01 12:00"], utc=True)
valid_times   = pd.date_range("2024-01-02", periods=24, freq="1h", tz="UTC")

rows = []
for kt in forecast_runs:
    for vt in valid_times:
        noise = rng.normal(0, 5)
        rows.append({"knowledge_time": kt, "valid_time": vt, "value": 80 + noise})

df_forecasts = pd.DataFrame(rows)
print(f"DataFrame shape: {df_forecasts.shape}")
df_forecasts.head(4)


DataFrame shape: (72, 3)


,knowledge_time,valid_time,value
0,2024-01-01 00:00:00+00:00,2024-01-02 00:00:00+00:00,84.357144
1,2024-01-01 00:00:00+00:00,2024-01-02 01:00:00+00:00,81.117978
2,2024-01-01 00:00:00+00:00,2024-01-02 02:00:00+00:00,83.394568
3,2024-01-01 00:00:00+00:00,2024-01-02 03:00:00+00:00,80.337895


In [6]:
ts_forecast = TimeSeries.from_pandas(
    df_forecasts,
    name="wind_power_forecast",
    unit="MW",
    frequency="1h",
    data_type=DataType.FORECAST,
    timeseries_type=TimeSeriesType.OVERLAPPING,
    location=GeoLocation(latitude=59.33, longitude=18.07),
    labels={"site": "Gotland", "model": "NWP-v2"},
)

print(ts_forecast)
print(f"\nArrow schema:\n{ts_forecast.table.schema}")

# to_pandas() restores a (knowledge_time, valid_time) MultiIndex
df_back = ts_forecast.to_pandas()
print(f"\nDataFrame index names : {df_back.index.names}")
df_back.head(6)

TimeSeries
┌────────────────────────────────────────────────────────────┐
│  Name:             wind_power_forecast                     │
│  Shape:            VERSIONED                               │
│  Rows:             72                                      │
│  Frequency:        1h                                      │
│  Timezone:         UTC                                     │
│  Unit:             MW                                      │
│  Data type:        FORECAST                                │
│  Location:         59.33°N, 18.07°E                        │
│  Timeseries type:  OVERLAPPING                             │
│  Labels:           {'site': 'Gotland', 'model': 'NWP-v2'}  │
├────────────────────────────────────────────────────────────┤
│                                      wind_power_forecast   │
│  2024-01-01 00:00, 2024-01-02 00:00              84.3571   │
│  2024-01-01 00:00, 2024-01-02 01:00               81.118   │
│  2024-01-01 00:00, 2024-01-02 02:00       

value
knowledge_time            valid_time                          
2024-01-01 00:00:00+00:00 2024-01-02 00:00:00+00:00  84.357144
                          2024-01-02 01:00:00+00:00  81.117978
                          2024-01-02 02:00:00+00:00  83.394568
                          2024-01-02 03:00:00+00:00  80.337895
                          2024-01-02 04:00:00+00:00  81.445597
                          2024-01-02 05:00:00+00:00  83.156441

In [7]:
# Inspect the latest forecast run via the pandas MultiIndex
latest_run = forecast_runs[-1]
df_latest = df_back.loc[latest_run]
print(f"Latest run ({latest_run.date()}) — first 4 rows:")
print(df_latest.head(4).to_string())


Latest run (2024-01-01) — first 4 rows:
                               value
valid_time                          
2024-01-02 00:00:00+00:00  71.563328
2024-01-02 01:00:00+00:00  72.764438
2024-01-02 02:00:00+00:00  73.386502
2024-01-02 03:00:00+00:00  75.013766


## 5. Shape 3 — `AUDIT`

The `AUDIT` shape is the full bi-temporal log: **every correction to every forecast run**.  
Each row is uniquely identified by `(knowledge_time, change_time, valid_time)`.

- `knowledge_time` — which forecast run the row belongs to  
- `change_time` — when the correction/update was recorded  
- `valid_time` — the wall-clock time the value applies to  

Optional extra columns: `changed_by`, `annotation`

Required columns: `knowledge_time`, `change_time`, `valid_time`, `value`

In [8]:
# One forecast run with two correction rounds
kt = pd.Timestamp("2024-01-01 00:00", tz="UTC")
valid_window = pd.date_range("2024-01-02", periods=6, freq="1h", tz="UTC")

# Original values
original = pd.DataFrame({
    "knowledge_time": kt,
    "change_time":    pd.Timestamp("2024-01-01 00:00", tz="UTC"),
    "valid_time":     valid_window,
    "value":          rng.uniform(70, 100, 6),
    "changed_by":     "model-auto",
    "annotation":     None,
})

# Analyst corrections for just 2 of the 6 hours
corrections = pd.DataFrame({
    "knowledge_time": kt,
    "change_time":    pd.Timestamp("2024-01-01 09:00", tz="UTC"),
    "valid_time":     valid_window[:2],
    "value":          [85.0, 90.0],
    "changed_by":     "alice",
    "annotation":     "manual correction after sensor check",
})

df_audit = pd.concat([original, corrections], ignore_index=True)

# AUDIT shape (change_time present) must use from_arrow() — from_pandas() only
# supports SIMPLE and VERSIONED shapes (the two shapes used for inserting data).
arrow_audit = pa.Table.from_pandas(df_audit, preserve_index=False)
ts_audit = TimeSeries.from_arrow(
    arrow_audit,
    name="wind_power_audit",
    unit="MW",
    data_type=DataType.FORECAST,
    timeseries_type=TimeSeriesType.OVERLAPPING,
)

print(ts_audit)
print(f"\nArrow schema:\n{ts_audit.table.schema}")

ts_audit.to_pandas()

TimeSeries
┌──────────────────────────────────────────────────────────────────────────┐
│  Name:             wind_power_audit                                      │
│  Shape:            AUDIT                                                 │
│  Rows:             8                                                     │
│  Timezone:         UTC                                                   │
│  Unit:             MW                                                    │
│  Data type:        FORECAST                                              │
│  Timeseries type:  OVERLAPPING                                           │
├──────────────────────────────────────────────────────────────────────────┤
│                                                        wind_power_audit  │
│  2024-01-01 00:00, 2024-01-01 00:00, 2024-01-02 00:00           87.5229  │
│  2024-01-01 00:00, 2024-01-01 00:00, 2024-01-02 01:00           89.4954  │
│  2024-01-01 00:00, 2024-01-01 00:00, 2024-01-02 02:00          

value  \
knowledge_time            change_time               valid_time                             
2024-01-01 00:00:00+00:00 2024-01-01 00:00:00+00:00 2024-01-02 00:00:00+00:00  87.522939   
                                                    2024-01-02 01:00:00+00:00  89.495398   
                                                    2024-01-02 02:00:00+00:00  72.533330   
                                                    2024-01-02 03:00:00+00:00  82.474222   
                                                    2024-01-02 04:00:00+00:00  71.248425   
                                                    2024-01-02 05:00:00+00:00  84.819725   
                          2024-01-01 09:00:00+00:00 2024-01-02 00:00:00+00:00  85.000000   
                                                    2024-01-02 01:00:00+00:00  90.000000   

                                                                               changed_by  \
knowledge_time            change_time               valid_time                              
2024-01-01 00:00:00+00:00 2024-01-01 00:00:00+00:00 2024-01-02 00:00:00+00:00  model-auto   
                                                    2024-01-02 01:00:00+00:00  model-auto   
                                                    2024-01-02 02:00:00+00:00  model-auto   
                                                    2024-01-02 03:00:00+00:00  model-auto   
                                                    2024-01-02 04:00:00+00:00  model-auto   
                                                    2024-01-02 05:00:00+00:00  model-auto   
                          2024-01-01 09:00:00+00:00 2024-01-02 00:00:00+00:00       alice   
                                                    2024-01-02 01:00:00+00:00       alice   

                                                                                                         annotation  
knowledge_time            change_time               valid_time                                                       
2024-01-01 00:00:00+00:00 2024-01-01 00:00:00+00:00 2024-01-02 00:00:00+00:00                                   NaN  
                                                    2024-01-02 01:00:00+00:00                                   NaN  
                                                    2024-01-02 02:00:00+00:00                                   NaN  
                                                    2024-01-02 03:00:00+00:00                                   NaN  
                                                    2024-01-02 04:00:00+00:00                                   NaN  
                                                    2024-01-02 05:00:00+00:00                                   NaN  
                          2024-01-01 09:00:00+00:00 2024-01-02 00:00:00+00:00  manual correction after sensor check  
                                                    2024-01-02 01:00:00+00:00  manual correction after sensor check

## 6. Metadata & `metadata_dict()`

Every `TimeSeries` carries a rich metadata payload that travels with the data.

In [9]:
import json

metadata = ts_actuals.metadata_dict()
print(json.dumps(metadata, indent=2))


{
  "name": "wind_power",
  "description": null,
  "unit": "MW",
  "labels": {
    "site": "Gotland",
    "asset_type": "wind"
  },
  "timezone": "Europe/Stockholm",
  "frequency": "1h",
  "data_type": "OBSERVATION",
  "location": {
    "longitude": 18.07,
    "latitude": 59.33
  },
  "timeseries_type": "FLAT",
  "shape": "SIMPLE",
  "num_rows": 24
}


In [10]:
# All metadata fields are also accessible as direct attributes
print(f"name            : {ts_actuals.name}")
print(f"unit            : {ts_actuals.unit}")
print(f"frequency       : {ts_actuals.frequency}")
print(f"data_type       : {ts_actuals.data_type}")
print(f"timeseries_type : {ts_actuals.timeseries_type}")
print(f"timezone        : {ts_actuals.timezone}")
print(f"location        : {ts_actuals.location}")
print(f"labels          : {ts_actuals.labels}")
print(f"shape           : {ts_actuals.shape}")
print(f"num_rows        : {ts_actuals.num_rows}")
print(f"len()           : {len(ts_actuals)}")


name            : wind_power
unit            : MW
frequency       : 1h
data_type       : OBSERVATION
timeseries_type : FLAT
timezone        : Europe/Stockholm
location        : GeoLocation(latitude=59.33, longitude=18.07)
labels          : {'site': 'Gotland', 'asset_type': 'wind'}
shape           : DataShape.SIMPLE
num_rows        : 24
len()           : 24


## 7. `from_arrow()` — Zero-Copy Path from PyArrow

When data arrives already as a `pa.Table` (e.g. read directly from PostgreSQL via `psycopg` with `fetch_arrow_table()` or via ADBC), use `from_arrow()`.  
No pandas conversion happens at all — this is the fastest possible ingestion path.

In [11]:
# Build the Arrow table directly (simulates a DB read)
ts_type = pa.timestamp("us", tz="UTC")
arrow_table = pa.table({
    "valid_time": pa.array(
        pd.date_range("2024-06-01", periods=48, freq="1h", tz="UTC").astype("int64"),
        type=ts_type,
    ),
    "value": pa.array(rng.uniform(40, 130, 48), type=pa.float64()),
})

print("Arrow table schema:")
print(arrow_table.schema)
print(f"\nRows: {len(arrow_table)}")

# Wrap with metadata — no data copy, no pandas round-trip
ts_arrow = TimeSeries.from_arrow(
    arrow_table,
    name="wind_power_arrow",
    unit="MW",
    frequency="1h",
    data_type=DataType.OBSERVATION,
    timeseries_type=TimeSeriesType.FLAT,
    location=GeoLocation(latitude=59.33, longitude=18.07),
)

print(f"\n{ts_arrow}")

# The .table property gives back the same Arrow object (no copy)
assert ts_arrow.table is arrow_table
print("\nts_arrow.table is the original arrow_table object (no copy confirmed)")

Arrow table schema:
valid_time: timestamp[us, tz=UTC]
value: double

Rows: 48

TimeSeries
┌──────────────────────────────────────┐
│  Name:             wind_power_arrow  │
│  Shape:            SIMPLE            │
│  Rows:             48                │
│  Frequency:        1h                │
│  Timezone:         UTC               │
│  Unit:             MW                │
│  Data type:        OBSERVATION       │
│  Location:         59.33°N, 18.07°E  │
├──────────────────────────────────────┤
│                    wind_power_arrow  │
│  2024-06-01 00:00           69.6875  │
│  2024-06-01 01:00           53.0072  │
│  2024-06-01 02:00           49.3063  │
│  ...                            ...  │
│  2024-06-02 21:00           86.3551  │
│  2024-06-02 22:00           63.0369  │
│  2024-06-02 23:00           124.244  │
└──────────────────────────────────────┘

ts_arrow.table is the original arrow_table object (no copy confirmed)


## 8. `to_pandas()` Round-Trip

`to_pandas()` converts back to a pandas DataFrame and restores the conventional index structure used throughout timedb's existing read API.

In [12]:
for ts, label in [
    (ts_actuals,  "SIMPLE   (index: valid_time)"),
    (ts_forecast, "VERSIONED (index: knowledge_time, valid_time)"),
    (ts_audit,    "AUDIT     (index: knowledge_time, change_time, valid_time)"),
]:
    df = ts.to_pandas()
    print(f"── {label}")
    print(f"   DataFrame.index.names : {df.index.names}")
    print(f"   DataFrame.columns     : {list(df.columns)}")
    print(f"   First row:")
    print(f"   {df.iloc[[0]].to_string()}")
    print()


── SIMPLE   (index: valid_time)
   DataFrame.index.names : ['valid_time']
   DataFrame.columns     : ['value']
   First row:
                                   value
valid_time                           
2024-01-01 00:00:00+00:00  104.176923

── VERSIONED (index: knowledge_time, valid_time)
   DataFrame.index.names : ['knowledge_time', 'valid_time']
   DataFrame.columns     : ['value']
   First row:
                                                            value
knowledge_time            valid_time                          
2024-01-01 00:00:00+00:00 2024-01-02 00:00:00+00:00  84.357144

── AUDIT     (index: knowledge_time, change_time, valid_time)
   DataFrame.index.names : ['knowledge_time', 'change_time', 'valid_time']
   DataFrame.columns     : ['value', 'changed_by', 'annotation']
   First row:
                                                                                      value  changed_by annotation
knowledge_time            change_time               valid_time           

## 9. Timezone Handling

All timestamps are stored internally as `pa.timestamp('us', tz='UTC')`.  
`from_pandas()` handles three cases:

| Input | Behaviour |
|---|---|
| Already UTC | No allocation — column left as-is |
| Non-UTC tz-aware (e.g. `Europe/Stockholm`) | `tz_convert("UTC")` — single new array |
| Naive (no tz) | `tz_localize("UTC")` + `UserWarning` |

The `timezone` metadata field is a **display hint** for the user's local domain and does not affect the stored data.

In [13]:
# ── Case 1: non-UTC timezone (Europe/Stockholm = UTC+1 in winter) ────────────
df_swe = pd.DataFrame({
    "valid_time": pd.date_range("2024-01-15 01:00", periods=6, freq="1h", tz="Europe/Stockholm"),
    "value": [10.0, 20.0, 30.0, 40.0, 50.0, 60.0],
})
ts_swe = TimeSeries.from_pandas(df_swe, name="se_price", unit="EUR/MWh", timezone="Europe/Stockholm")
print("Input tz: Europe/Stockholm  →  stored as UTC")
print(ts_swe.to_pandas().head(3).to_string())
print()

# ── Case 2: naive timestamps (warns) ─────────────────────────────────────────
df_naive = pd.DataFrame({
    "valid_time": pd.date_range("2024-03-01", periods=4, freq="1h"),  # no tz
    "value": [1.0, 2.0, 3.0, 4.0],
})
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    ts_naive = TimeSeries.from_pandas(df_naive, name="naive_example", unit="MW")
    print(f"Warning issued: '{w[0].message}'")

# Internal type is always UTC regardless
assert ts_naive.table.schema.field("valid_time").type == pa.timestamp("us", tz="UTC")
print("Internal Arrow type confirmed: timestamp[us, tz=UTC]")


Input tz: Europe/Stockholm  →  stored as UTC
                           value
valid_time                      
2024-01-15 00:00:00+00:00   10.0
2024-01-15 01:00:00+00:00   20.0
2024-01-15 02:00:00+00:00   30.0

Warning issued: 'Column 'valid_time' has no timezone; assuming UTC.'
Internal Arrow type confirmed: timestamp[us, tz=UTC]


## 10. Performance — Ingesting a Large DataFrame

The optimised `from_pandas()` pipeline avoids O(rows) full copies.  
Here we benchmark ingest time for 1 million rows and compare against a naïve double-copy baseline.

In [14]:
N = 1_000_000

df_large = pd.DataFrame({
    "valid_time": pd.date_range("2020-01-01", periods=N, freq="1min", tz="UTC"),
    "value":      rng.uniform(0, 500, N),
})

print(f"DataFrame size : {N:,} rows  ({df_large.memory_usage(deep=True).sum() / 1e6:.1f} MB)")
print()

# Warm up
_ = TimeSeries.from_pandas(df_large, name="warmup", unit="MW")

# Benchmark from_pandas
RUNS = 5
times = []
for _ in range(RUNS):
    t0 = time.perf_counter()
    ts_large = TimeSeries.from_pandas(df_large, name="large_series", unit="MW", frequency="1min")
    times.append(time.perf_counter() - t0)

mean_ms = np.mean(times) * 1000
print(f"from_pandas()  — {RUNS} runs — mean {mean_ms:.1f} ms  (min {min(times)*1000:.1f} ms)")
print(f"Rows/second    : {N / np.mean(times):,.0f}")
print()

# from_arrow is even faster (no pandas involved)
arrow_large = ts_large.table

times_arrow = []
for _ in range(RUNS):
    t0 = time.perf_counter()
    ts_from_arrow = TimeSeries.from_arrow(arrow_large, name="large_from_arrow", unit="MW")
    times_arrow.append(time.perf_counter() - t0)

mean_arrow_ms = np.mean(times_arrow) * 1000
print(f"from_arrow()   — {RUNS} runs — mean {mean_arrow_ms:.2f} ms  (validation only, no data copy)")
print(f"Speedup vs from_pandas : {mean_ms / mean_arrow_ms:.0f}x")


DataFrame size : 1,000,000 rows  (16.0 MB)

from_pandas()  — 5 runs — mean 5.5 ms  (min 4.1 ms)
Rows/second    : 181,292,776

from_arrow()   — 5 runs — mean 0.01 ms  (validation only, no data copy)
Speedup vs from_pandas : 479x
